In [10]:
%%writefile GridLoadSimulator.py
"""
GridLoadSimulator.py
Generates simulated grid load data and outputs it to a CSV file.

Output file:
    live_grid_load.csv

Columns produced:
    Timestamp
    Load_Percent
    Temperature_C
    Season
    Day_Type
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def simulate_grid_load(
    start="2024-01-01 00:00",
    periods=24*30,  # 30 days hourly data
    freq="h",
):
    # Create timestamps
    timestamps = pd.date_range(start=start, periods=periods, freq=freq)

    # Seasonal temperature patterns
    temps = []
    loads = []

    for ts in timestamps:
        # Basic seasonal temperature pattern
        month = ts.month
        if month in [12, 1, 2]:  # Winter
            base_temp = np.random.normal(0, 5)
        elif month in [3, 4, 5]:  # Spring
            base_temp = np.random.normal(10, 5)
        elif month in [6, 7, 8]:  # Summer
            base_temp = np.random.normal(27, 4)
        else:  # Fall
            base_temp = np.random.normal(14, 5)

        # Hourly variations in load
        hour = ts.hour
        if 0 <= hour <= 6:
            base_load = np.random.normal(40, 5)
        elif 7 <= hour <= 15:
            base_load = np.random.normal(55, 7)
        elif 16 <= hour <= 21:
            base_load = np.random.normal(75, 8)
        else:
            base_load = np.random.normal(55, 5)

        # Temperature influence on load:
        temp_factor = base_temp * 0.25
        final_load = max(0, min(100, base_load + temp_factor))

        temps.append(base_temp)
        loads.append(final_load)

    # Determine season label
    def season_of(ts):
        m = ts.month
        if m in [12, 1, 2]: return "Winter"
        if m in [3, 4, 5]: return "Spring"
        if m in [6, 7, 8]: return "Summer"
        return "Fall"

    # Determine weekday/weekend label
    def day_type(ts):
        return "Weekend" if ts.weekday() >= 5 else "Weekday"

    df = pd.DataFrame({
        "Timestamp": timestamps,
        "Load_Percent": loads,
        "Temperature_C": temps,
        "Season": [season_of(ts) for ts in timestamps],
        "Day_Type": [day_type(ts) for ts in timestamps],
    })

    return df

def generate_csv(output_path="live_grid_load.csv"):
    df = simulate_grid_load()
    df.to_csv(output_path, index=False)
    print(f"Grid load CSV successfully created: {output_path}")

if __name__ == "__main__":
    generate_csv()

Overwriting GridLoadSimulator.py


In [11]:
from GridLoadSimulator import generate_csv
generate_csv()

Grid load CSV successfully created: live_grid_load.csv


C:\Users\e40014066\Desktop\CPE 551-Phyton\Group Project\GridLoadSimulator.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  timestamps = pd.date_range(start=start, periods=periods, freq=freq)


In [3]:
%%writefile data_loader.py
"""
Module responsible for loading, cleaning and merging all project datasets:
-Grid load data
-Weather Data
-Electrical outages
Module name: data_loader.py
"""
import pandas as pd
from datetime import datetime
import numpy as np

class DataLoader:
  """
  Handles reading CSV files, cleaning raw values, and merging datasets.
  """
  def __init__(self, grid_path: str, weather_path: str, outage_path: str):
        self.grid_path = grid_path
        self.weather_path = weather_path
        self.outage_path = outage_path
  
  # Grid Load
  def load_grid_data(self):
    """
    Load the simulated grid data.
    """
    df = pd.read_csv(self.grid_path)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df = df.dropna(subset=['Timestamp'])
    return df

  # Weather Data Cleaning
  def _parse_noaa_numeric(self, val: str):
        """
        Extract the numeric portion for values like '+0001,1' or '99999,9'.      
        """
        if isinstance(val, str):
            try:
                raw = val.split(',')[0].replace('+', '')
                num = int(raw)
                if num > 9000:  # missing value indicator
                    return np.nan
                return num / 10.0  #TMP often stored as tenths of degrees C
            except:
                return np.nan
        return np.nan

  def load_weather_data(self) -> pd.DataFrame:
        """
        Load weather data and convert key fields.
        """
        df = pd.read_csv(self.weather_path)
        df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
        df = df.dropna(subset=['DATE'])

        # Parse temperature
        if 'TMP' in df.columns:
            df['temperature_C'] = df['TMP'].apply(self._parse_noaa_numeric)
        else:
            df['temperature_C'] = np.nan

        # Parse dew point
        if 'DEW' in df.columns:
            df['dewpoint_C'] = df['DEW'].apply(self._parse_noaa_numeric)
        else:
            df['dewpoint_C'] = np.nan

        # Parse sea-level pressure
        if 'SLP (Sea Level Pressure)' in df.columns:
            df['pressure_hPa'] = df['SLP (Sea Level Pressure)'].apply(self._parse_noaa_numeric)
        else:
            df['pressure_hPa'] = np.nan

        return df[['DATE', 'temperature_C', 'dewpoint_C', 'pressure_hPa']]

  # Electrical Outage Data
  def load_outage_data(self):
    """
    Load historic electrical outage data for Hudson County.
    """
    df = pd.read_csv(self.outage_path)
    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df = df.dropna(subset=['start_time'])
    return df
    
  # Merging  
  def merge_all(self):
    """
    Merge grid, weather and outage data.
    """
    grid = self.load_grid_data().sort_values('Timestamp')
    weather = self.load_weather_data().sort_values('DATE')
    merged = pd.merge_asof(
            grid,
            weather,
            left_on='Timestamp',
            right_on='DATE',
            direction='nearest'
        )

    merged.drop(columns=['DATE'], inplace=True)

    # Add simple outage indicator based on nearest outage events
    outages = self.load_outage_data()
    outages = outages.sort_values('start_time')

    merged['nearest_outage_customers'] = 0
    merged['nearest_outage_duration'] = 0.0

    idx = 0
    for i, row in merged.iterrows():
        ts = row['Timestamp']
        # Find closest outage in history
        while idx + 1 < len(outages) and outages.iloc[idx + 1]['start_time'] < ts:
             idx += 1
        outage = outages.iloc[idx]
        merged.at[i, 'nearest_outage_customers'] = outage['mean_customers']
        merged.at[i, 'nearest_outage_duration'] = outage['duration']

    return merged

Overwriting data_loader.py


In [4]:
%%writefile power_station.py
"""
Represents a Hudson County power station with load history and capacity.
Module name: power_station.py
"""
class PowerStation:
    def __init__(self, name: str, latitude: float, longitude: float, rated_capacity: float):
        self.name = name
        self.latitude = latitude
        self.longitude = longitude
        self.rated_capacity = rated_capacity
        self.load_history = []

    def add_load(self, load_percent: float):
        self.load_history.append(load_percent)

    def compute_average_load(self):
        if not self.load_history:
            return 0.0
        return sum(self.load_history) / len(self.load_history)

    def compute_peak_load(self):
        return max(self.load_history) if self.load_history else 0.0

    def __str__(self):
        return f"{self.name} @({self.latitude},{self.longitude}) capacity={self.rated_capacity}%"

    def __lt__(self, other):
        return self.rated_capacity < other.rated_capacity


Overwriting power_station.py


In [5]:
%%writefile load_forecaster.py
"""
Implements simple short-term forecasting: rolling mean and seasonal weighting.
Module name: load_forecaster.py
"""
import pandas as pd

class LoadForecaster:
    def __init__(self, merged_df: pd.DataFrame):
        self.df = merged_df.copy()
        self.df = self.df.sort_values('Timestamp')

    def prepare_features(self):
        self.df['hour'] = self.df['Timestamp'].dt.hour
        self.df['weekday'] = self.df['Timestamp'].dt.weekday
        return self.df

    def forecast_next_hour(self):
        """
        Uses last N load values for rolling average forecasting.
        """
        recent = self.df['Load_Percent'].tail(6)  # last ~1 hour of 10-minute data
        return recent.mean()

    def forecast_next_24h(self):
        """
        Projects next 24 hours based on mean of each hour of day historically.
        """
        self.df['hour'] = self.df['Timestamp'].dt.hour
        hour_profile = self.df.groupby('hour')['Load_Percent'].mean()

        forecast = []
        for h in range(24):
            forecast.append(hour_profile.get(h, hour_profile.mean()))

        return pd.Series(forecast)

Overwriting load_forecaster.py


In [1]:
%%writefile outage_analyzer.py
"""
Computes outage frequency patterns, basic probabilities, and severity scores.
Module name: outage_analyzer.py
"""
import pandas as pd
import numpy as np

class OutageAnalyzer:
    def __init__(self, outage_df: pd.DataFrame):
        self.outages = outage_df.copy()
        self.outages['hour'] = self.outages['start_time'].dt.hour
        self.outages['month'] = self.outages['start_time'].dt.month

    def outages_by_season(self):
        return self.outages.groupby('month').size()

    def compute_outage_probability(self, timestamp):
        """
        Probability = (# of outages at same hour)/total outages
        """
        hour = timestamp.hour
        total = len(self.outages)
        occ = len(self.outages[self.outages['hour'] == hour])
        return occ / total if total > 0 else 0.0

    def recent_outages(self, timestamp, window_hours=6):
        """
        Returns outages in the last X hours.
        """
        start = timestamp - pd.Timedelta(hours=window_hours)
        return self.outages[(self.outages['start_time'] >= start) &
                            (self.outages['start_time'] <= timestamp)]

Overwriting outage_analyzer.py


In [7]:
%%writefile risk_model.py
"""
Estimates risk from load, outage probability, and weather.
Module name: risk_model.py
"""
class RiskModel:
    def __init__(self, station_capacity: float):
        self.capacity = station_capacity

    def compute_risk_score(self, load_percent, outage_prob, temperature):
        """
        Weighted scoring model. Adjust freely.
        """
        load_factor = load_percent / self.capacity
        temp_factor = temperature / 40.0 if temperature is not None else 0.0
        risk = 0.6*load_factor + 0.25*outage_prob + 0.15*temp_factor
        return risk

    def classify(self, score):
        if score < 0.3:
            return "Low"
        elif score < 0.6:
            return "Moderate"
        elif score < 0.85:
            return "High"
        return "Critical"


Overwriting risk_model.py


In [8]:
%%writefile utils.py
"""
Contains generator, decorators, and set operations.
Module name: utils.py
"""
import time
from functools import wraps

def runtime_logger(func):
    """
    Decorator logging runtime of any function.
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        out = func(*args, **kwargs)
        print(f"{func.__name__} executed in {time.time() - start:.4f} sec")
        return out
    return wrapper

def hourly_load_generator(df):
    """
    Streams load entries row-by-row.
    """
    for _, row in df.iterrows():
        yield row

def unique_outage_days(outage_df):
    """
    Example set operation.
    """
    return set(outage_df['start_time'].dt.date)

Overwriting utils.py


In [9]:
from data_loader import DataLoader
from power_station import PowerStation
from load_forecaster import LoadForecaster
from outage_analyzer import OutageAnalyzer
from risk_model import RiskModel

loader = DataLoader("live_grid_load.csv", "Weather data for Hudson station.csv", "Project electrical outages data.csv")
merged = loader.merge_all()

station = PowerStation("Hudson Substation", 40.728, -74.078, rated_capacity=85)
station.load_history = merged['Load_Percent'].tolist()

forecaster = LoadForecaster(merged)
forecast_hour = forecaster.forecast_next_hour()

outages = loader.load_outage_data()
out_analyzer = OutageAnalyzer(outages)
out_prob = out_analyzer.compute_outage_probability(merged['Timestamp'].iloc[-1])

risk_engine = RiskModel(station.rated_capacity)
risk_score = risk_engine.compute_risk_score(forecast_hour, out_prob, merged['temperature_C'].iloc[-1])
risk_level = risk_engine.classify(risk_score)

print("Next-hour forecast:", forecast_hour)
print("Outage probability:", out_prob)
print("Risk score:", risk_score, "=>", risk_level)

Next-hour forecast: 68.71209104379007
Outage probability: 0.03749043611323642
Risk score: 0.49214913404329785 => Moderate
